# HAR model for SPY volatility
We forecast next-week realized variance with the HAR model of Corsi (2009) and compare it to simple benchmarks. This cell loads the libraries and the project modules.

In [1]:
%load_ext autoreload
%autoreload 2
import sys; sys.path.append("../src")
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

from volforecast.data import load
from volforecast.estimators import daily_variance
from volforecast.features import make_dataset
from volforecast.models import HARModel, NaiveModel, MeanModel
from volforecast.evaluate import mse, qlike

## Building the dataset
The features are the daily, weekly (5-day) and monthly (22-day) averages of daily variance. The target is the sum of variance over the next 5 trading days.

In [2]:
X, y = make_dataset(load("SPY"))

Let's check by hand the code of make_dataset: lets pick a date, for example 18-10-2016

In [3]:
d = "2016-10-18"
v = daily_variance(load("SPY"), method="gk", include_overnight=True)

# Computed by hand from v, using explicit date windows (.loc includes both ends)
manual = pd.Series({
    "rv_d":   v.loc[d],
    "rv_w":   v.loc["2016-10-12":d].mean(),           # Oct 12, 13, 14, 17, 18
    "rv_m":   v.loc["2016-09-19":d].mean(),           # 22 trading days
    "target": v.loc["2016-10-19":"2016-10-25"].sum(), # Oct 19, 20, 21, 24, 25
})

# Computed by make_dataset
from_pipeline = pd.concat([X.loc[d], pd.Series({"target": y.loc[d]})])

comparison = pd.DataFrame({"manual": manual, "pipeline": from_pipeline})
comparison["difference"] = comparison["manual"] - comparison["pipeline"]
print(comparison)

# Sanity check on the window sizes
print("days in rv_w window:", len(v.loc["2016-10-12":d]))                     # expect 5
print("days in rv_m window:", len(v.loc["2016-09-19":d]))                     # expect 22
print("days in target window:", len(v.loc["2016-10-19":"2016-10-25"]))        # expect 5

          manual  pipeline    difference
rv_d    0.000085  0.000085  0.000000e+00
rv_w    0.000055  0.000055  1.626303e-19
rv_m    0.000042  0.000042  1.355253e-20
target  0.000098  0.000098 -1.734723e-18
days in rv_w window: 5
days in rv_m window: 22
days in target window: 5


## Train / test split
We train on data up to the end of 2016 and test from 2017 onwards. The split is chronological, so the model never sees future data.

In [4]:
X_train, y_train = X.loc[:"2016"], y.loc[:"2016"]
X_test, y_test = X.loc["2017":], y.loc["2017":]
print(f"Train: {len(X_train)} days")
print(f"Test: {len(X_test)} days")

Train: 2496 days
Test: 2477 days


## Fitting the HAR model
The model is a linear regression of the target on the three features plus an intercept, fitted by OLS.

In [5]:
har = HARModel().fit(X_train, y_train)
har.coef_

beta_0    0.000152
beta_d    0.770363
beta_w    1.219424
beta_m    2.172540
dtype: float64

We check our coefficients against statsmodels OLS. The differences should be close to zero.

In [6]:
ols = sm.OLS(y_train, sm.add_constant(X_train)).fit()

coef_check = pd.DataFrame({
    "mine": har.coef_.to_numpy(),
    "statsmodels": ols.params.to_numpy(),
}, index=har.coef_.index)
coef_check["difference"] = coef_check["mine"] - coef_check["statsmodels"]
print(coef_check)

max_diff = (har.predict(X_train) - ols.fittedvalues).abs().max()
print("max difference in fitted values:", max_diff)

            mine  statsmodels    difference
beta_0  0.000152     0.000152  2.439455e-19
beta_d  0.770363     0.770363  7.771561e-16
beta_w  1.219424     1.219424 -3.330669e-15
beta_m  2.172540     2.172540 -2.220446e-15
max difference in fitted values: 2.0816681711721685e-17


## Benchmarks
The naive model assumes the next 5 days look like the last 5 (5 × rv_w), and the mean model predicts the training average. HAR has to beat both to be useful.

In [7]:
naive = NaiveModel().fit(X_train, y_train)
mean = MeanModel().fit(X_train, y_train)

## Out-of-sample evaluation
We compare the models on the test set with MSE and QLIKE. The ratio columns show each loss relative to the naive model, so a value below 1 means it does better.

In [8]:
y_har = har.predict(X_test)
y_naive = naive.predict(X_test)
y_mean = mean.predict(X_test)
table = pd.DataFrame({
    'MSE' : [mse(y_test, y_naive), mse(y_test, y_mean), mse(y_test, y_har)],
    'QLIKE' : [qlike(y_test, y_naive), qlike(y_test, y_mean), qlike(y_test, y_har)]
}, index=['naive', 'mean', 'HAR'])
table["MSE vs naive"] = table["MSE"] / table.loc["naive", "MSE"]
table["QLIKE vs naive"] = table["QLIKE"] / table.loc["naive", "QLIKE"]
table

,MSE,QLIKE,MSE vs naive,QLIKE vs naive
naive,0.000003,0.372034,1.000000,1.000000
mean,0.000004,0.782536,1.349857,2.103400
HAR,0.000002,0.292511,0.823562,0.786248


## All five assets
We wrap the whole pipeline in one function, `evaluate_asset(ticker)`, so it runs the same way on every asset. One change from the SPY analysis above: we drop the last 5 training rows. Their targets sum variance over the first days of 2017, which is test data, so keeping them would leak a little of the test period into training.

In [9]:
from volforecast.data import TICKERS

HORIZON = 5


def evaluate_asset(ticker):
    """Run the full HAR pipeline on one asset.

    Returns the results table (3 models x MSE, QLIKE and both ratios vs naive)
    and HAR's fitted coefficients.
    """
    X, y = make_dataset(load(ticker), horizon=HORIZON)

    # Chronological split. The last HORIZON training targets sum variance from
    # early 2017, so we drop those rows to keep the test period out of training.
    X_train, y_train = X.loc[:"2016"].iloc[:-HORIZON], y.loc[:"2016"].iloc[:-HORIZON]
    X_test, y_test = X.loc["2017":], y.loc["2017":]

    models = {
        "naive": NaiveModel(horizon=HORIZON),
        "mean": MeanModel(),
        "HAR": HARModel(),
    }
    rows = {}
    for name, model in models.items():
        y_hat = model.fit(X_train, y_train).predict(X_test)
        rows[name] = {"MSE": mse(y_test, y_hat), "QLIKE": qlike(y_test, y_hat)}

    table = pd.DataFrame(rows).T
    table["MSE vs naive"] = table["MSE"] / table.loc["naive", "MSE"]
    table["QLIKE vs naive"] = table["QLIKE"] / table.loc["naive", "QLIKE"]
    return table, models["HAR"].coef_


# Sanity check on SPY. The numbers differ slightly from the table above
# because the last 5 training rows are now dropped.
spy_table, spy_coef = evaluate_asset("SPY")
spy_table

,MSE,QLIKE,MSE vs naive,QLIKE vs naive
naive,0.000003,0.372034,1.000000,1.000000
mean,0.000004,0.782988,1.350111,2.104614
HAR,0.000002,0.292579,0.823557,0.786431


We loop over the five assets and collect two summary tables: HAR's loss ratios against the naive model, and HAR's coefficients.

In [10]:
results, coefs = {}, {}
for ticker in TICKERS:
    results[ticker], coefs[ticker] = evaluate_asset(ticker)

scores = pd.DataFrame({
    ticker: table.loc["HAR", ["MSE vs naive", "QLIKE vs naive"]]
    for ticker, table in results.items()
}).T.add_prefix("HAR ")
coefficients = pd.DataFrame(coefs).T

print("Scores (below 1 = HAR beats naive)")
display(scores)
print("HAR coefficients")
display(coefficients)

Scores (below 1 = HAR beats naive)


,HAR MSE vs naive,HAR QLIKE vs naive
SPY,0.823557,0.786431
TLT,1.248298,0.839986
FXE,0.881838,0.635442
GLD,0.676414,0.692315
USO,0.686629,0.734613


HAR coefficients


,beta_0,beta_d,beta_w,beta_m
SPY,0.000152,0.770365,1.219393,2.172412
TLT,0.000123,0.390822,0.285807,2.850608
FXE,0.000033,0.056795,0.137722,4.016607
GLD,0.000129,0.222126,0.746586,3.188881
USO,0.000254,0.293810,0.744645,3.453533
